In [ ]:
!pip install datasets transformers

In [1]:
!pip install -U datasets huggingface_hub fsspec

  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)


In [15]:
!pip install -U unsloth

  Using cached unsloth-2026.2.1-py3-none-any.whl.metadata (69 kB)
  Using cached unsloth_zoo-2026.2.1-py3-none-any.whl.metadata (32 kB)
  Using cached xformers-0.0.34-cp39-abi3-manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached bitsandbytes-0.49.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvid

In [2]:
!nvidia-smi

Mon Feb 16 16:55:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
pip install --upgrade wandb

In [4]:
import os

from google.colab import userdata
os.environ['HF_TOKEN']=userdata.get('HF_TOKEN')

In [5]:
from datasets import Dataset, load_dataset

ds = load_dataset("newfacade/LeetCodeDataset")
ds

DatasetDict({
    train: Dataset({
        features: ['task_id', 'question_id', 'difficulty', 'tags', 'problem_description', 'starter_code', 'estimated_date', 'prompt', 'completion', 'entry_point', 'test', 'input_output', 'query', 'response'],
        num_rows: 2641
    })
    test: Dataset({
        features: ['task_id', 'question_id', 'difficulty', 'tags', 'problem_description', 'starter_code', 'estimated_date', 'prompt', 'completion', 'entry_point', 'test', 'input_output', 'query', 'response'],
        num_rows: 228
    })
})

In [6]:
cols = ds['train'].column_names
columns_needed_to_dropped=[]
for col in cols:
    if col=='problem_description' or col=='response':
        print(type(col))
        continue
    else:
        columns_needed_to_dropped.append(col)
columns_needed_to_dropped

<class 'str'>
<class 'str'>


['task_id',
 'question_id',
 'difficulty',
 'tags',
 'starter_code',
 'estimated_date',
 'prompt',
 'completion',
 'entry_point',
 'test',
 'input_output',
 'query']

In [7]:
ds = ds.remove_columns(columns_needed_to_dropped)
ds

DatasetDict({
    train: Dataset({
        features: ['problem_description', 'response'],
        num_rows: 2641
    })
    test: Dataset({
        features: ['problem_description', 'response'],
        num_rows: 228
    })
})

In [8]:
def format_data(example):
    return {
        "text": f"""### Instruction:
Solve the leetcode question for the provided problem description by clear explanation and then practical python code.

### Input:
{example['problem_description']}

### Response:
{example['response']}"""
    }


In [9]:
train_dataset = ds['train'].map(format_data)
train_dataset

Map:   0%|          | 0/2641 [00:00<?, ? examples/s]

Dataset({
    features: ['problem_description', 'response', 'text'],
    num_rows: 2641
})

In [10]:
!du -h -d 1 /root | sort -hr | head -20

3.6G	/root
3.2G	/root/.julia
301M	/root/.cache
135M	/root/.npm
2.4M	/root/.launchpadlib
142K	/root/.ipython
67K	/root/.jupyter
59K	/root/.local
13K	/root/.config
8.0K	/root/.keras


In [11]:
!rm -rf /root/.cache/huggingface


In [12]:
!rm -rf /root/.cache/pip


In [13]:
!rm -rf checkpoint-*


In [11]:
from huggingface_hub import login
login()


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [10]:
from unsloth import FastLanguageModel
import torch

model_id = "deepseek-ai/deepseek-coder-1.3b-base"

max_seq_length = 1024
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit
)

/tmp/ipython-input-10-2002019851.py:1: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
deepseek-ai/deepseek-coder-1.3b-base does not have a padding token! Will use pad_token = <pad>.


In [ ]:
!pip install modelscope
import os; os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 70.5 MB/s eta 0:00:00


In [11]:
tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


Unsloth 2026.2.1 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


In [13]:
print(''.join(train_dataset['text'][0]))


    ### Instruction: Solve the leetcode question for the provided problem description by clear explaination and then practical python code for the question.
    ### Input:
    Given an array of integers nums and an integer target, return indices of the two numbers such that they add up to target.
You may assume that each input would have exactly one solution, and you may not use the same element twice.
You can return the answer in any order.
 
Example 1:

Input: nums = [2,7,11,15], target = 9
Output: [0,1]
Explanation: Because nums[0] + nums[1] == 9, we return [0, 1].

Example 2:

Input: nums = [3,2,4], target = 6
Output: [1,2]

Example 3:

Input: nums = [3,3], target = 6
Output: [0,1]

 
Constraints:

2 <= nums.length <= 104
-109 <= nums[i] <= 109
-109 <= target <= 109
Only one valid answer exists.

 
Follow-up: Can you come up with an algorithm that is less than O(n2) time complexity?
    ### Response:
    To solve this problem efficiently, we can use a hash map (dictionary in Pytho

In [21]:
!pip install codebleu

In [12]:
test_dataset = ds['test'].map(format_data)
test_dataset

Map:   0%|          | 0/228 [00:00<?, ? examples/s]

Dataset({
    features: ['problem_description', 'response', 'text'],
    num_rows: 228
})

In [ ]:
!pip install evaluate

In [25]:
from codebleu import calc_codebleu
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Replace -100 in labels (ignore index)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Decode
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute CodeBLEU
    result = calc_codebleu(
        references=[[ref] for ref in decoded_labels],
        predictions=decoded_preds,
        lang="python"
    )

    return {
        "codebleu": result["codebleu"],
        "ngram_match": result["ngram_match_score"],
        "weighted_ngram_match": result["weighted_ngram_match_score"],
        "syntax_match": result["syntax_match_score"],
        "dataflow_match": result["dataflow_match_score"],
    }


In [13]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

training_arguments = TrainingArguments(
    per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,

)

In [15]:
import wandb
wandb.init(mode="disabled")


In [16]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    max_seq_length = max_seq_length,
    dataset_text_feild = "text",
    args = training_arguments,

)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,641 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,991,360 of 1,361,463,296 (1.10% trained)


Step,Training Loss
10,12.480700
20,4.677800
30,2.173200
40,1.068100
50,0.836500
60,0.681100
70,0.607500
80,0.605800
90,0.580100
100,0.586100


TrainOutput(global_step=100, training_loss=2.4296946048736574, metrics={'train_runtime': 341.2395, 'train_samples_per_second': 2.344, 'train_steps_per_second': 0.293, 'total_flos': 5607348821385216.0, 'train_loss': 2.4296946048736574, 'epoch': 0.3028009084027252})

In [36]:
model.eval()

test_dataset = test_dataset.select(range(5))
predictions = []
references = []

for example in test_dataset:
    prompt = f"""### Instruction:
Solve the leetcode question for the provided problem description by clear explanation and then practical python code.

### Input:
{example['problem_description']}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.0
        )

    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated = decoded_output.split("### Response:")[-1].strip()

    predictions.append(generated)
    references.append(example["response"])


In [39]:
!pip uninstall tree-sitter tree-sitter-python -y
!pip install tree-sitter==0.20.1
!pip install tree-sitter-python==0.20.0


Found existing installation: tree-sitter 0.22.3
Uninstalling tree-sitter-0.22.3:
  Successfully uninstalled tree-sitter-0.22.3
Found existing installation: tree-sitter-python 0.25.0
Uninstalling tree-sitter-python-0.25.0:
  Successfully uninstalled tree-sitter-python-0.25.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for tree-sitter: filename=tree_sitter-0.20.1-cp311-cp311-linux_x86_64.whl size=425850 sha256=c6ec2fa81d09fcb100377b7f7abc7dd46b023ca014b1425a9461d4ac5878a8a0
  Stored in directory: /root/.cache/pip/wheels/6c/be/3a/6841c52111041475b3c693b347ac03f305330cbf7fb1c6365d
Successfully built tree-sitter
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
codebleu 0.7.0 require

In [25]:
example = train_dataset[0]
print(''.join(example['problem_description']))

Given an array of integers nums and an integer target, return indices of the two numbers such that they add up to target.
You may assume that each input would have exactly one solution, and you may not use the same element twice.
You can return the answer in any order.
 
Example 1:

Input: nums = [2,7,11,15], target = 9
Output: [0,1]
Explanation: Because nums[0] + nums[1] == 9, we return [0, 1].

Example 2:

Input: nums = [3,2,4], target = 6
Output: [1,2]

Example 3:

Input: nums = [3,3], target = 6
Output: [0,1]

 
Constraints:

2 <= nums.length <= 104
-109 <= nums[i] <= 109
-109 <= target <= 109
Only one valid answer exists.

 
Follow-up: Can you come up with an algorithm that is less than O(n2) time complexity?


In [28]:
model.eval()

example = train_dataset[0]

prompt = f"""### Instruction:
Solve the leetcode question for the provided problem description by clear explanation and then practical python code and finally end up with explaination.

### Input:
{example['problem_description']}

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=502,
        temperature=0.2,
        top_p=0.9,
        do_sample=False,
        use_cache=True
    )

output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(output_text)


### Instruction:
Solve the leetcode question for the provided problem description by clear explanation and then practical python code and finally end up with explaination.

### Input:
Given an array of integers nums and an integer target, return indices of the two numbers such that they add up to target.
You may assume that each input would have exactly one solution, and you may not use the same element twice.
You can return the answer in any order.
 
Example 1:

Input: nums = [2,7,11,15], target = 9
Output: [0,1]
Explanation: Because nums[0] + nums[1] == 9, we return [0, 1].

Example 2:

Input: nums = [3,2,4], target = 6
Output: [1,2]

Example 3:

Input: nums = [3,3], target = 6
Output: [0,1]

 
Constraints:

2 <= nums.length <= 104
-109 <= nums[i] <= 109
-109 <= target <= 109
Only one valid answer exists.

 
Follow-up: Can you come up with an algorithm that is less than O(n2) time complexity?

### Response:
```python
from typing import List

class Solution:
    def twoSum(self, nums:

In [29]:
model = model.merge_and_unload()
model.save_pretrained("leetcode_full_model")
tokenizer.save_pretrained("leetcode_full_model")


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


('leetcode_full_model/tokenizer_config.json',
 'leetcode_full_model/special_tokens_map.json',
 'leetcode_full_model/tokenizer.json')

In [30]:
from huggingface_hub import login
login()


In [32]:
model.push_to_hub("gajasankarraja/leetcode-deepseek-1.3b")
tokenizer.push_to_hub("gajasankarraja/leetcode-deepseek-1.3b")

README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pp9_wmn/model.safetensors:   0%|          | 61.9kB /  891MB            

Saved model to https://huggingface.co/gajasankarraja/leetcode-deepseek-1.3b


README.md:   0%|          | 0.00/574 [00:00<?, ?B/s]